# 7.2 Fehleranalyse und optimale Schrittweite

In Abschnitt 7.1 haben wir den Fehler der drei Differenzenformeln im
Log-Log-Plot dargestellt. Dabei haben wir Schrittweiten bis $10^{-10}$
betrachtet. Was passiert, wenn wir $\Delta x$ noch viel kleiner machen?
Wir starten direkt mit einem Experiment.

## Lernziele

* [ ] Sie können den **Abschneidefehler der Vorwärtsdifferenz** mithilfe der
  Taylor-Entwicklung herleiten und die Fehlerordnung $O(\Delta x)$ angeben.
* [ ] Sie können erklären, warum der zentrale Differenzenquotient die
  **Fehlerordnung $O(\Delta x^2)$** erreicht.
* [ ] Sie wissen, was **Maschinengenauigkeit** ist, und können erklären, warum
  sehr kleine Schrittweiten den Fehler wieder vergrößern.
* [ ] Sie können die **optimale Schrittweite** für die Vorwärtsdifferenz und
  den zentralen Differenzenquotienten der Größenordnung nach angeben.

## Einstieg: Ein überraschender Befund

Wir erweitern den Log-Log-Plot aus Abschnitt 7.1 und verlängern den
Bereich der Schrittweiten bis $\Delta x = 10^{-16}$:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Funktion und analytische Ableitung
def f(x):
    return np.sin(x)

def f_strich(x):
    return np.cos(x)

# Differenzenformeln aus Abschnitt 7.1
def vorwaerts(f, x, dx):
    return (f(x + dx) - f(x)) / dx

def zentral(f, x, dx):
    return (f(x + dx) - f(x - dx)) / (2 * dx)

# Auswertestelle und exakter Wert
x0    = 1.0
exakt = f_strich(x0)

# 200 Schrittweiten gleichmäßig im Log-Maßstab von 0.1 bis 1e-16
dx_werte = np.logspace(-1, -16, 200)

# Fehler für jede Schrittweite berechnen und speichern
fehler_v = []
fehler_z = []
for dx in dx_werte:
    fehler_v.append(abs(exakt - vorwaerts(f, x0, dx)))
    fehler_z.append(abs(exakt - zentral(f, x0, dx)))

# Log-Log-Plot: beide Achsen logarithmisch
fig, ax = plt.subplots()
ax.loglog(dx_werte, fehler_v, label='Vorwärtsdifferenz')
ax.loglog(dx_werte, fehler_z, label='Zentraler Differenzenquot.')
ax.set_xlabel('Schrittweite Δx')
ax.set_ylabel('Fehler |f′(x₀) − Näherung|')
ax.set_title('Fehler für sehr kleine Schrittweiten')
ax.legend()
ax.grid(True, which='both')
plt.show()

Für sehr kleine Schrittweiten steigt der Fehler wieder an, anstatt
weiter zu fallen. Es gibt also eine optimale Schrittweite. Diesen Befund
erklären wir in zwei Schritten: Zunächst analysieren wir mit der
Taylor-Entwicklung, warum der Fehler bei größeren $\Delta x$ abnimmt und
warum die zentrale Formel dabei besser abschneidet. Dann untersuchen wir,
was bei sehr kleinen $\Delta x$ passiert.

## Abschneidefehler: Warum die Formeln unterschiedlich genau sind

Die **Taylor-Entwicklung** schreibt den Funktionswert $f(x + \Delta x)$
als unendliche Reihe um den Punkt $x$:

\begin{equation*}
f(x + \Delta x)
= f(x) + \Delta x \cdot f'(x) +
  \frac{\Delta x^2}{2} \cdot f''(x) +
  \frac{\Delta x^3}{6} \cdot f'''(x) + \cdots
\end{equation*}

Wir lösen nach $f'(x)$ auf. Dazu subtrahieren wir $f(x)$ auf beiden
Seiten und teilen durch $\Delta x$:

\begin{equation*}
f'(x)
= \frac{f(x + \Delta x) - f(x)}{\Delta x} - \frac{\Delta x}{2} \cdot f''(x) -
  \frac{\Delta x^2}{6} \cdot f'''(x) - \cdots
\end{equation*}

Der erste Term rechts ist genau unsere Vorwärtsdifferenz. Die übrigen
Terme sind der Fehler, der entsteht, wenn wir die unendliche Reihe nach
dem ersten Glied abschneiden. Diesen Fehler nennen wir **Abschneidefehler**
(englisch: truncation error).

Der führende Fehlerterm ist $\frac{\Delta x}{2} \cdot |f''(x)|$. Er ist
proportional zu $\Delta x$. Wir schreiben das als **Fehlerordnung**
$O(\Delta x)$: halbieren wir $\Delta x$, halbiert sich auch der Fehler.
Das haben wir in Abschnitt 7.1 in der Fehlertabelle genau so beobachtet.

### Warum der zentrale Differenzenquotient besser ist

Für den zentralen Differenzenquotienten brauchen wir zwei
Taylor-Entwicklungen:

\begin{align*}
f(x + \Delta x)
&= f(x) + \Delta x \cdot f'(x) +
   \frac{\Delta x^2}{2} \cdot f''(x) +
   \frac{\Delta x^3}{6} \cdot f'''(x) + \cdots \\[6pt]
f(x - \Delta x)
&= f(x) - \Delta x \cdot f'(x) +
   \frac{\Delta x^2}{2} \cdot f''(x) -
   \frac{\Delta x^3}{6} \cdot f'''(x) + \cdots
\end{align*}

Wir subtrahieren die zweite von der ersten Gleichung:

\begin{equation*}
f(x + \Delta x) - f(x - \Delta x)
= 2 \Delta x \cdot f'(x) + \frac{\Delta x^3}{3} \cdot f'''(x) + \cdots
\end{equation*}

Die $f''$-Terme heben sich durch die Symmetrie exakt auf. Teilen wir
durch $2 \Delta x$, ergibt sich:

\begin{equation*}
f'(x)
= \frac{f(x + \Delta x) - f(x - \Delta x)}{2 \Delta x} -
  \frac{\Delta x^2}{6} \cdot f'''(x) - \cdots
\end{equation*}

Der führende Fehlerterm ist $\frac{\Delta x^2}{6} \cdot |f'''(x)|$,
also Fehlerordnung $O(\Delta x^2)$. Halbieren wir $\Delta x$, wird der
Fehler viermal kleiner. Das ist der Grund für die steilere Gerade im
Log-Log-Plot.

### Fehlerordnungen im Überblick

| Formel | Führender Fehlerterm | Fehlerordnung |
| --- | --- | --- |
| Vorwärtsdifferenz | $\frac{\Delta x}{2} \cdot \lvert f''(x)\rvert$ | $O(\Delta x)$ |
| Rückwärtsdifferenz | $\frac{\Delta x}{2} \cdot \lvert f''(x)\rvert$ | $O(\Delta x)$ |
| Zentraler Differenzenquot. | $\frac{\Delta x^2}{6} \cdot \lvert f'''(x)\rvert$ | $O(\Delta x^2)$ |

Wir überprüfen die Fehlerordnungen numerisch. Dazu vergleichen wir den
tatsächlichen Fehler mit dem theoretisch vorhergesagten:

In [ ]:
import numpy as np

# Funktion, Ableitung, zweite und dritte Ableitung von sin(x)
def f(x):
    return np.sin(x)

def f_strich(x):
    return np.cos(x)

def f_zweimal(x):
    """Zweite Ableitung von sin(x): -sin(x)."""
    return -np.sin(x)

def f_dreimal(x):
    """Dritte Ableitung von sin(x): -cos(x)."""
    return -np.cos(x)

def vorwaerts(f, x, dx):
    return (f(x + dx) - f(x)) / dx

def zentral(f, x, dx):
    return (f(x + dx) - f(x - dx)) / (2 * dx)

x0    = 1.0
exakt = f_strich(x0)

# Vergleich für die Vorwärtsdifferenz:
# Theoretischer Fehler laut Taylor: (dx/2) * |f''(x0)|
print("Vorwärtsdifferenz:")
for dx in [0.1, 0.01, 0.001]:
    fehler_num  = abs(exakt - vorwaerts(f, x0, dx))
    fehler_theo = (dx / 2) * abs(f_zweimal(x0))
    print(f"  dx = {dx:.3f}: tatsächlicher Fehler = {fehler_num:.2e}, "
          f"Theorie = {fehler_theo:.2e}")

print()

# Vergleich für den zentralen Differenzenquotienten:
# Theoretischer Fehler laut Taylor: (dx^2/6) * |f'''(x0)|
print("Zentraler Differenzenquotient:")
for dx in [0.1, 0.01, 0.001]:
    fehler_num  = abs(exakt - zentral(f, x0, dx))
    fehler_theo = (dx**2 / 6) * abs(f_dreimal(x0))
    print(f"  dx = {dx:.3f}: tatsächlicher Fehler = {fehler_num:.2e}, "
          f"Theorie = {fehler_theo:.2e}")

Die theoretischen Vorhersagen stimmen gut mit den tatsächlichen Fehlern
überein. Die Taylor-Entwicklung beschreibt das Verhalten der
Differenzenformeln also korrekt.

### Mini-Übung 1

1. Die Rückwärtsdifferenz lautet
   $f'(x) \approx \frac{f(x) - f(x - \Delta x)}{\Delta x}$.
   Entwickeln Sie $f(x - \Delta x)$ nach Taylor und leiten Sie den
   führenden Fehlerterm her. Was stellen Sie im Vergleich zur
   Vorwärtsdifferenz fest?
2. Warum heben sich die $f''$-Terme bei der zentralen Formel auf, bei
   der Vorwärtsdifferenz aber nicht? Begründen Sie in zwei Sätzen ohne
   Rechnung.
3. Der Fehler des zentralen Differenzenquotienten ist proportional zu
   $\Delta x^2$. Verdoppeln wir $\Delta x$ von 0.01 auf 0.02, um wie
   viel wächst der Fehler dann ungefähr? Überprüfen Sie Ihre Antwort
   mit Python.

In [ ]:
# Code-Zelle

## Rundungsfehler und optimale Schrittweite

Der Abschneidefehler sinkt mit kleiner werdendem $\Delta x$. Aber der
Einstiegsplot zeigt, dass der Gesamtfehler irgendwann wieder ansteigt.
Die Ursache ist der **Rundungsfehler**.

Computer speichern Fließkommazahlen nicht exakt. Der relative Fehler
einer einzelnen Zahl ist durch die **Maschinengenauigkeit**
$\varepsilon_M$ begrenzt. Für `float64` gilt
$\varepsilon_M \approx 2.2 \cdot 10^{-16}$, was etwa 16 gültigen
Dezimalstellen entspricht:

In [ ]:
import numpy as np

# np.finfo(float) liefert Informationen über den float64-Datentyp.
# eps ist die kleinste Zahl, für die der Computer 1.0 + eps != 1.0 erkennt.
eps = np.finfo(float).eps
print(f"Maschinengenauigkeit: {eps:.2e}")

Das Problem bei sehr kleinen $\Delta x$ ist die **Auslöschung**. Im
Zähler $f(x + \Delta x) - f(x)$ subtrahieren wir zwei fast gleich große
Zahlen. Die führenden Stellen heben sich auf, und die hinteren Stellen
sind durch Rundungsfehler verfälscht. Je kleiner $\Delta x$, desto mehr
gültige Stellen gehen verloren.

Wir können das Zusammenspiel von Abschneide- und Rundungsfehler
näherungsweise durch ein einfaches Modell beschreiben. Dabei interessieren
uns vor allem die Abhängigkeiten von $\Delta x$; konstante Faktoren und
die genaue Größe von $f(x)$, $f''(x)$ fassen wir als Größenordnung~1
zusammen. Heuristisch schreiben wir den Gesamtfehler als

\begin{equation*}
E(\Delta x)
\approx \underbrace{\frac{\Delta x}{2} \cdot |f''(x)|}_{\text{Abschneidefehler}} +
\underbrace{\frac{\varepsilon_M}{\Delta x} \cdot |f(x)|}_{\text{Rundungsfehler}}
\end{equation*}

Diese Gleichung ist also kein exakter Ausdruck, sondern ein
**Fehlermodell**, das die wesentliche Abhängigkeit von $\Delta x$ (und
groben Größenordnungen von $f$ und seinen Ableitungen) erfasst.

Das Minimum dieser modellhaften Summe liegt bei der **optimalen
Schrittweite**. Wenn wir für die Vorwärtsdifferenz nur die Abhängigkeit
von $\Delta x$ betrachten und annehmen, dass $|f(x)|$ und $|f''(x)|$
in der Größenordnung~1 liegen, erhalten wir

\begin{equation*}
\Delta x_\text{opt}
\approx \sqrt{\varepsilon_M}
\approx \sqrt{2.2 \cdot 10^{-16}}
\approx 1.5 \cdot 10^{-8}
\end{equation*}

Für den zentralen Differenzenquotienten ist der Abschneidefehler
proportional zu $\Delta x^2$. Die analoge Rechnung (wieder bis auf
Faktoren der Größenordnung~1) liefert

\begin{equation*}
\Delta x_\text{opt}
\approx \varepsilon_M^{1/3}
\approx (2.2 \cdot 10^{-16})^{1/3}
\approx 6 \cdot 10^{-6}
\end{equation*}

Streng genommen hängt die optimale Schrittweite also sowohl von
$\varepsilon_M$ als auch von $|f(x)|$ und den Ableitungen ab; für viele
"normale" Funktionen liegen diese Beiträge aber in der Größenordnung~1,
sodass in erster Linie die Maschinengenauigkeit den Maßstab vorgibt.

### Mini-Übung 2

1. Erstellen Sie den Einstiegsplot neu, aber diesmal mit allen drei
   Differenzenformeln (Vorwärts, Rückwärts, Zentral) im Bereich
   $\Delta x \in [10^{-1}, 10^{-16}]$. Beschriften Sie alle drei Kurven.
2. Lesen Sie die optimale Schrittweite für Vorwärtsdifferenz und
   zentralen Differenzenquotienten aus dem Plot ab. Berechnen Sie
   anschließend `np.sqrt(np.finfo(float).eps)` und
   `np.finfo(float).eps ** (1/3)` und vergleichen Sie mit Ihrer Ablesung.
3. Führen Sie dieselbe Analyse für $g(x) = \exp(x)$ an der Stelle
   $x_0 = 0$ durch. Ändert sich die optimale Schrittweite?

In [ ]:
# Code-Zelle

## Zusammenfassung und Ausblick

Die Taylor-Entwicklung erklärt, warum die drei Differenzenformeln
unterschiedlich genau sind. Vorwärts- und Rückwärtsdifferenz haben den
Abschneidefehler $O(\Delta x)$: ihr Fehler ist proportional zu $\Delta x$.
Der zentrale Differenzenquotient erreicht $O(\Delta x^2)$, weil der
$f''$-Term durch Symmetrie herausfällt. Für sehr kleine Schrittweiten
dominiert die Auslöschung den Gesamtfehler. Die optimalen Schrittweiten
liegen bei $\Delta x \approx \sqrt{\varepsilon_M} \approx 1.5 \cdot 10^{-8}$
für die Vorwärtsdifferenz und bei
$\Delta x \approx \varepsilon_M^{1/3} \approx 6 \cdot 10^{-6}$
für den zentralen Differenzenquotienten.

Im Übungskapitel wenden wir die Differenzenformeln auf reale Messdaten
an. Dort liegen die Funktionswerte nicht mehr an frei wählbaren Stellen
vor, sondern sind durch den Sensor fest vorgegeben.